# MLP


In [70]:
import numpy as np

class Sigmoid:
    def __init__(self):
        self.input = None
        self.output = None
        self.grad = None
    def forward(self, x, grad=False):
        self.input = x
        x = np.clip(x, -500, 500)
        self.output = 1 / (1 + np.exp(-x))
        if grad: self.grad=self.output*(1-self.output)
        return self.output

class MSELoss:
    def __init__(self, preds, targets, grad=False):
        n = preds.size
        self.loss = np.sum(np.square(preds - targets)) / n

        # dL / d(preds)
        if grad:
            self.grad = 2 * (preds - targets) / n
        else:
            self.grad = None

class Perceptron:
    def __init__(self, input_dim=1, output_dim=1, activation=Sigmoid):

        # weight shape: (output_dim, input_dim)
        self.weight = np.random.randn(output_dim, input_dim) * 0.01

        # bias shape: (output_dim, 1)
        self.bias = np.zeros((output_dim, 1))
        self.activation = activation()

        self.input = None
        self.z = None
        self.output = None

    def forward(self, x, grad=False):
        """
        x shape:
            (input_dim, batch_size)

        return:
            (output_dim, batch_size)
        """
        self.input = x
        # z = Wx + b
        self.z = np.matmul(self.weight, x) + self.bias
        # a = sigmoid(z)
        self.output = self.activation.forward(self.z, grad=grad)
        return self.output

    def backprop(self, grad_output, lr=0.01):
        dz = grad_output * self.activation.grad
        dw = np.matmul(dz, self.input.T) 
        db = np.sum(dz, axis=1, keepdims=True)
        dx = np.matmul(self.weight.T, dz)

        self.weight -= lr * dw
        self.bias -= lr * db

        return dx

    def predict(self, x, threshold=0.5):
        """
        이진 분류용 predict
        """
        output = self.forward(x)
        return (output >= threshold).astype(int)

In [75]:
def train_single_step(model, x, y, lr=0.1):
    preds = model.forward(x, grad=True)
    loss = MSELoss(preds, y, grad=True)
    model.backprop(loss.grad, lr)
    


train_x = np.array([
    [0.1, 0.2, 0.8, 0.9]
])  # shape: (1, 4)

train_y = np.array([
    [0, 0, 1, 1]
])  # shape: (1, 4)
model = Perceptron(
    input_dim=1,
    output_dim=1,
    activation=Sigmoid
)
print(model.forward(train_x))
for i in range(300):
    train_single_step(model, train_x, train_y, lr=2)
    if i%100==0:
        print(f"{i+1}: ",model.forward(train_x))
    

[[0.49970095 0.4994019  0.49760761 0.49730856]]
1:  [[0.50447596 0.50857909 0.53315428 0.53723777]]
101:  [[0.12103744 0.18770526 0.83766037 0.89646729]]
201:  [[0.07428497 0.13328253 0.88393711 0.93587632]]
